In [ ]:
import os
import gc
import cv2
import pyraws
import warnings
import numpy as np
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning, module="pyraws")

raw_data_dir = '/home/laura/Scriptie/ruwe_data'
bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/THRawS'
temp_extract_dir = '/home/laura/Scriptie/data/temp_thraws/L0_Data_Uitgepakt'

os.makedirs(bewerkte_data_dir, exist_ok=True)

In [ ]:
PATCH_SIZE = 256
STRIDE = 256

all_folders = [d for d in os.listdir(temp_extract_dir) if os.path.isdir(os.path.join(temp_extract_dir, d))]
file_labels = [0 if "NE" in folder else 1 for folder in all_folders]

train_folders, test_folders, train_labels, test_labels = train_test_split(
    all_folders, file_labels, test_size=0.2, random_state=42, stratify=file_labels
)

print(f" Train files : {len(train_folders)}")
print(f" Test files : {len(test_folders)}")

def folder_to_feature(folder_list):
    features = []
    labels = []

    for folder in folder_list:
        full_event_path = os.path.join(temp_extract_dir, folder)
        label = 0 if "NE" in folder else 1  # 0 = not fire, 1 = fire

        try:
            raw_data = pyraws.Raw_event()
            raw_data.from_path(full_event_path, ['B04', 'B08', 'B11'])
            tile = raw_data.get_granule(0)

            def get_numpy_image(band_name, tile):
                band_data = tile.get_band(band_name)
                matrix = band_data.pixels if hasattr(band_data, 'pixels') else band_data
                if hasattr(matrix, 'cpu'): matrix = matrix.cpu().numpy()
                return matrix

            b04 = get_numpy_image('B04', tile)
            b08 = get_numpy_image('B08', tile)
            b11 = get_numpy_image('B11', tile)

            b11_resized = cv2.resize(b11, (b04.shape[1], b04.shape[0]),
                                     interpolation=cv2.INTER_NEAREST)
            X_tile = np.stack([b04, b08, b11_resized], axis=-1)

            del b04, b08, b11, b11_resized
            gc.collect()

            for y in range(0, X_tile.shape[0] - PATCH_SIZE + 1, STRIDE):
                for x in range(0, X_tile.shape[1] - PATCH_SIZE + 1, STRIDE):
                    patch = X_tile[y:y+PATCH_SIZE, x:x+PATCH_SIZE, :]

                    b04_mean = np.mean(patch[:, :, 0])
                    b08_mean = np.mean(patch[:, :, 1])
                    b11_mean = np.mean(patch[:, :, 2])

                    nbr  = (b08_mean - b11_mean) / (b08_mean + b11_mean + 1e-6)
                    ndvi = (b08_mean - b04_mean) / (b08_mean + b04_mean + 1e-6)

                    features.append([
                        b11_mean,                        # Mean B11
                        np.std(patch[:, :, 2]),          # STD B11
                        np.max(patch[:, :, 0]),          # Max B04
                        b11_mean / (b08_mean + 1),       # Ratio B11/B08
                        nbr,                             # Normalized Burn Ratio
                        ndvi                             # Normalized Difference Vegetation Index
                    ])
                    labels.append(label)

            del X_tile
            gc.collect()

        except Exception as e:
            print(f"Error processing {folder}: {e}")

    return np.array(features), np.array(labels)

In [ ]:
from sklearn.utils import shuffle

X_train_tabular, y_train = folder_to_feature(train_folders)
X_test_tabular,  y_test  = folder_to_feature(test_folders)

# Shuffle to mix fire and non-fire patches
X_train, y_train = shuffle(X_train_tabular, y_train, random_state=42)
X_test,  y_test  = shuffle(X_test_tabular,  y_test,  random_state=42)

print(f"Train — 0: {(y_train==0).sum()}, 1: {(y_train==1).sum()}")
print(f"Test  — 0: {(y_test==0).sum()},  1: {(y_test==1).sum()}")

np.save(os.path.join(bewerkte_data_dir, 'X_train_THRawS.npy'), X_train)
np.save(os.path.join(bewerkte_data_dir, 'y_train_THRawS.npy'), y_train)
np.save(os.path.join(bewerkte_data_dir, 'X_test_THRawS.npy'),  X_test)
np.save(os.path.join(bewerkte_data_dir, 'y_test_THRawS.npy'),  y_test)

print(f"Train set: {X_train.shape[0]} patches, {X_train.shape[1]} features")
print(f"Test set:  {X_test.shape[0]} patches")

In [ ]:
# Check the distribution of folders in train and test sets (more events than non events)
all_folders = [d for d in os.listdir(temp_extract_dir) if os.path.isdir(os.path.join(temp_extract_dir, d))]
ne_folders    = [f for f in all_folders if "NE" in f]
fire_folders  = [f for f in all_folders if "NE" not in f]

print(f"NE folders (geen brand): {len(ne_folders)}")
print(f"Brand folders:           {len(fire_folders)}")
print(f"\nNE folders: {ne_folders}")
print(f"Brand folders: {fire_folders}")

In [ ]:
'''PATCH_SIZE = 256
STRIDE = 256

all_folders = [d for d in os.listdir(temp_extract_dir) if os.path.isdir(os.path.join(temp_extract_dir, d))]
file_labels = [0 if "NE" in folder else 1 for folder in all_folders]

train_folders, test_folders, train_labels, test_labels = train_test_split(
    all_folders, file_labels, test_size=0.2, random_state=42, stratify=file_labels
)

print(f" Train files : {len(train_folders)}")
print(f" Test files : {len(test_folders)}")

def folder_to_feature(folder_list):
    features = []
    labels = []

    for folder in folder_list:
        full_event_path = os.path.join(temp_extract_dir, folder)
        label = 0 if "NE" in folder else 1  # 0 = not fire, 1 = fire

        try:
            raw_data = pyraws.Raw_event()
            raw_data.from_path(full_event_path, ['B04', 'B08', 'B11'])
            tile = raw_data.get_granule(0)

            def get_numpy_image(band_name, tile):
                band_data = tile.get_band(band_name)
                matrix = band_data.pixels if hasattr(band_data, 'pixels') else band_data
                if hasattr(matrix, 'cpu'): matrix = matrix.cpu().numpy()
                return matrix

            b04 = get_numpy_image('B04', tile)
            b08 = get_numpy_image('B08', tile)
            b11 = get_numpy_image('B11', tile)

            b11_resized = cv2.resize(b11, (b04.shape[1], b04.shape[0]),
                                     interpolation=cv2.INTER_NEAREST)
            X_tile = np.stack([b04, b08, b11_resized], axis=-1)

            del b04, b08, b11, b11_resized
            gc.collect()

            for y in range(0, X_tile.shape[0] - PATCH_SIZE + 1, STRIDE):
                for x in range(0, X_tile.shape[1] - PATCH_SIZE + 1, STRIDE):
                    patch = X_tile[y:y+PATCH_SIZE, x:x+PATCH_SIZE, :]

                    b04_mean = np.mean(patch[:, :, 0])
                    b08_mean = np.mean(patch[:, :, 1])
                    b11_mean = np.mean(patch[:, :, 2])

                    nbr  = (b08_mean - b11_mean) / (b08_mean + b11_mean + 1e-6)
                    ndvi = (b08_mean - b04_mean) / (b08_mean + b04_mean + 1e-6)

                    features.append([
                        b11_mean,                        # Mean B11
                        np.std(patch[:, :, 2]),          # STD B11
                        np.max(patch[:, :, 0]),          # Max B04
                        b11_mean / (b08_mean + 1),       # Ratio B11/B08
                        nbr,                             # Normalized Burn Ratio
                        ndvi                             # Normalized Difference Vegetation Index
                    ])
                    labels.append(label)

            del X_tile
            gc.collect()

        except Exception as e:
            print(f"Error processing {folder}: {e}")

    return np.array(features), np.array(labels)'''

In [ ]:
'''X_train_tabular, y_train = folder_to_feature(train_folders)
X_test_tabular, y_test = folder_to_feature(test_folders)

from sklearn.utils import shuffle

X_train, y_train = shuffle(X_train_tabular, y_train, random_state=42)
X_test, y_test   = shuffle(X_test_tabular,  y_test,  random_state=42)

print(f"Train — 0: {(y_train==0).sum()}, 1: {(y_train==1).sum()}")
print(f"Test  — 0: {(y_test==0).sum()},  1: {(y_test==1).sum()}")

#y_train = 1 - y_train
#y_test  = 1 - y_test

np.save(os.path.join(bewerkte_data_dir, 'X_train_THRawS.npy'), X_train)   # geshuffeld
np.save(os.path.join(bewerkte_data_dir, 'y_train_THRawS.npy'), y_train)
np.save(os.path.join(bewerkte_data_dir, 'X_test_THRawS.npy'), X_test)    # geshuffeld
np.save(os.path.join(bewerkte_data_dir, 'y_test_THRawS.npy'), y_test)

print(f"Train set: {X_train_tabular.shape[0]} patches")
print(f"Test set:  {X_test_tabular.shape[0]} patches")'''